In [34]:
import os
os.environ["PYTORCH_ENABLE_MPS_FALLBACK"] = "1"

import numpy as np
from evedesign.system import System, Protein
from evedesign.models.boltzfold import BoltzFoldTransformer
from evedesign.utils import ensure_sequence

In [35]:
seq = "TSENPLLALREKISALDEKLLALLAERRELAVEVGKAKLLSHRPVRDIDRERDLLERLITLGKAHHLDAHYITRLFQLIIEDSVLTQQALLQQH"
s = System([Protein(rep=seq, id='EcCM', first_index=2)])
inst = s.rep_to_instance()

In [36]:
from evedesign.tools.mmseqs2 import add_sequences_mmseqs2

s = add_sequences_mmseqs2(s, use_pairing=True)

m = BoltzFoldTransformer(device='cpu', use_msa=True, diffusion_samples=1)
m.build(s)

In [37]:
output_structures = m.transform([inst])

Processing 1 inputs with 1 threads.


  0%|          | 0/1 [00:00<?, ?it/s]

Generating MSA for /var/folders/xb/vj5zk8_x0bs4xmp72n7_6dbm0000gn/T/boltzfold_i6b2j2kf/inputs/instance_0.yaml with 1 protein entities.
Calling MSA server for target instance_0 with 1 sequences
MSA server URL: https://api.colabfold.com
MSA pairing strategy: greedy
No authentication provided for MSA server


100%|██████████| 1/1 [00:04<00:00,  4.99s/it]
/Users/khbelahsen/Documents/GitHub/work/marks/evedesign/.venv/lib/python3.12/site-packages/pytorch_lightning/utilities/migration/utils.py:56: The loaded checkpoint was produced with Lightning v2.5.0.post0, which is newer than your current Lightning version: v2.5.0
2026-04-27 15:45:46.174 | INFO     | evedesign.models.boltzfold:_load_model:208 - Boltz-2 loaded from /Users/khbelahsen/.boltz/boltz2_conf.ckpt
2026-04-27 15:56:20.251 | INFO     | evedesign.models.boltzfold:transform:371 - Boltz-2 output written to: /var/folders/xb/vj5zk8_x0bs4xmp72n7_6dbm0000gn/T/boltzfold_i6b2j2kf/predictions
2026-04-27 15:56:20.274 | INFO     | evedesign.models.boltzfold:transform:372 - Files written (6):
2026-04-27 15:56:20.276 | INFO     | evedesign.models.boltzfold:transform:375 -   instance_0/confidence_instance_0_model_0.json (443 bytes)
2026-04-27 15:56:20.277 | INFO     | evedesign.models.boltzfold:transform:375 -   instance_0/instance_0_model_0.cif (67

In [38]:
result = output_structures[0]
print(f"Score (boltz2 confidence score): {result.score}")

Score (boltz2 confidence score): 0.9250097274780273


In [ ]:
print("Confidence scores:")
for key, value in result.metadata.items():
    print(f"  {key}: {value}")
ei = result[0]
structures = ensure_sequence(ei.models["model_0"])

if ei.models:
    chain_id = structures[0].chains()[0]
    structure = structures[0]              
    print(f"\nChain: {chain_id}")
    print(f"Atom count: {len(structure.atom_array)}")
    print(f"Residue range: {structure.atom_array.res_id.min()} - {structure.atom_array.res_id.max()}")
    print(structure.atom_df().head(5))

Confidence scores:
  boltz_confidence: {'confidence_score': 0.9250097274780273, 'ptm': 0.8662264347076416, 'iptm': 0.0, 'ligand_iptm': 0.0, 'protein_iptm': 0.0, 'complex_plddt': 0.9397055506706238, 'complex_iplddt': 0.9397055506706238, 'complex_pde': 0.3475845456123352, 'complex_ipde': 0.0, 'chains_ptm': {'0': 0.8662264347076416}, 'pair_chains_iptm': {'0': {'0': 0.8662264347076416}}}

Chain: A
Atom count: 762
Residue range: 2 - 95


  chain_id  res_id ins_code res_name  hetero atom_name element  atom_id  \
0        A       2               THR   False         N       N        1   
1        A       2               THR   False        CA       C        2   
2        A       2               THR   False         C       C        3   
3        A       2               THR   False         O       O        4   
4        A       2               THR   False        CB       C        5   

   b_factor  occupancy  charge          x          y        z  
0    63.828        1.0       0 -24.150070  27.673620  3.00524  
1    63.828        1.0       0 -23.413990  26.426081  2.81203  
2    63.828        1.0       0 -24.291321  25.343229  2.19014  
3    63.828        1.0       0 -24.016069  24.151800  2.32021  
4    63.828        1.0       0 -22.862940  25.896330  4.14067  


In [40]:
! pip install py3Dmol -q

In [41]:
import io
import py3Dmol

ei = result[0]
if ei.models:
    chain_id = list(ei.models.keys())[0]
    structure = ei.models[chain_id]

    buf = io.StringIO()
    structure.to_file(buf, format="cif")
    cif_content = buf.getvalue()

    view = py3Dmol.view(width=800, height=500)
    view.addModel(cif_content, "cif")
    view.setStyle({
        "cartoon": {
            "colorscheme": {
                "prop": "b",
                "gradient": "roygb",
                "min": 50,
                "max": 90
            }
        }
    })
    view.zoomTo()
    view.show()

3Dmol.js failed to load for some reason. Please check your browser console for error messages.